# Baseline modeling

**GOAL:**

- Train multiple models
- compare their performance and metrics
- select the promising model

In [1]:
# Imports

import pandas as pd
import numpy as np

In [2]:
# load X_train, X_val, X_test, y_train, y_val, y_test

X_train = pd.read_csv(r'..\Data\processed\modeling\X_train.csv')
X_val = pd.read_csv(r'..\Data\processed\modeling\X_val.csv')
X_test = pd.read_csv(r'..\Data\processed\modeling\X_test.csv')

y_train = pd.read_csv(r'..\Data\processed\modeling\y_train.csv')
y_val = pd.read_csv(r'..\Data\processed\modeling\y_val.csv')
y_test = pd.read_csv(r'..\Data\processed\modeling\y_test.csv')

## 1: Baseline model

- linear regression

**comparison models**

- ridge regression
- random forest
- XG boost

**Comparison**

- compare by RMSE Rsqt

In [3]:
# helper functions
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error

all_results = {}
# all_results = pd.DataFrame(results).T
train_vs_val_result = {}

def log_metrics(y_val, y_pred):

    MAE = mean_absolute_error(y_val, y_pred)
    RMSE = root_mean_squared_error(y_val, y_pred)
    R2 = r2_score(y_val, y_pred)

    return MAE, RMSE, R2

def actual_metrics(y_val, y_pred):

    ACTUAL_Y_VALIDATION = np.expm1(y_val)
    ACTUAL_Y_PRED = np.expm1(y_pred)

    ACUTAL_MAE = mean_absolute_error(ACTUAL_Y_VALIDATION, ACTUAL_Y_PRED)
    ACTUAL_RMSE = root_mean_squared_error(ACTUAL_Y_VALIDATION, ACTUAL_Y_PRED)
    ACTUAL_R2 = r2_score(ACTUAL_Y_VALIDATION, ACTUAL_Y_PRED)

    return ACUTAL_MAE, ACTUAL_RMSE, ACTUAL_R2

def result(model_name, model, X_val, y_val):

    y_pred = model.predict(X_val)
    MAE, RMSE, R2 = log_metrics(y_val, y_pred)
    A_MAE, A_RMSE, A_R2 = actual_metrics(y_val, y_pred)

    all_results[model_name] = {
        'Log MAE': round(MAE, 4),
        'Log RMSE': round(RMSE, 4),
        'Log R2': round(R2, 4),
        'Actual MAE': round(A_MAE, 4),
        'Actual RMSE': round(A_RMSE, 4),
        'Actual R2': round(A_R2, 4)
    }

# def check_fitting(train_r2, val_r2):

#     fitting = ''

#     if train_r2 >= 0.90 and (val_r2 <= 0.50 or val_r2 < 0.60):
#             fitting = 'Over Fitting'

#     elif train_r2 < 0.30 and val_r2 < 0.30:
#         fitting = 'Under Fitting'
#     elif 0.70 <= train_r2 < 0.90 and 0.50 <= val_r2 < 0.80:
#         fitting = 'Normal'

#     elif 0.50 <= train_r2 < 0.70 and 0.30 <= val_r2 < 0.60:
#             fitting = 'Normal'

#     elif 0.20 <= train_r2 < 0.50 and 0.20 <= val_r2 < 0.50:
#         fitting = 'Weak'

#     return fitting

# Enhanced version of the above
def check_fitting(train_r2, val_r2):
    gap = train_r2 - val_r2

    # overfit gap is the primary signal, not absolute train score
    if gap > 0.15:
        return 'Over Fitting'

    # underfit both scores are low
    if train_r2 < 0.40 and val_r2 < 0.40:
        return 'Under Fitting'

    # weak model learned something but not enough
    if val_r2 < 0.50:
        return 'Weak'

    # good fit low gap, decent val score
    if val_r2 >= 0.50 and gap <= 0.15:
        return 'Normal'

    return 'Unknown'
            
def train_vs_val(model_name, model, X_train, X_val, y_train, y_val):
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_r2 = round(r2_score(y_train, train_pred), 4)
    val_r2 = round(r2_score(y_val, val_pred), 4)

    fitting = check_fitting(train_r2, val_r2)

    train_vs_val_result[model_name] = {
        'Train Score': r2_score(y_train, train_pred),
        'Val Score': r2_score(y_val, val_pred),
        'Fitting': fitting
    }

In [4]:
# 1.1 Baseline model selcetion (Linear regression)
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

result('Linear Regression', lr, X_val, y_val)
pd.DataFrame(all_results).T

,Log MAE,Log RMSE,Log R2,Actual MAE,Actual RMSE,Actual R2
Linear Regression,0.1994,0.303,0.3345,1754.5806,4462.086,0.1646


In [5]:
train_vs_val('Linear Regression', lr, X_train, X_val, y_train, y_val)
pd.DataFrame(train_vs_val_result).T

,Train Score,Val Score,Fitting
Linear Regression,0.462222,0.334473,Weak


# Baseline Model Report

## MAE - Error Metric
- Mean Absolute Error in actual price,
    - tells me that my regression model's predictions are **on average, ~1754 rs** away from the actual rent

## RMSE - Error Metric
- RMSE tells a gap,
    ```md
    RMSE = 4462
    MAE  = 1754
    Gap  = 2708  ← this is large
    ```
    - This gap means my model has some bad predictions (outliers), that pulls up RMSE score
    - Most predictions are around ₹1,754 off (MAE)
    - But a few are severely bad pulling RMSE up to ₹4,462
    - The gap (₹2,708) just tells us how much those outliers are skewing the score

## R2 Score - Eval metric

- R2 Score tells me that my baseline model (Linear Regression), can explains about ~33% of variation in the rent
    - so Since R2 score was low, 0.33 is weak for this dataset
- ** why R2 score is low?**
    - Linear regression can not predict non linear predictions

# Final Observation: Linear Regression Baseline

---

| Metric | Score | Indication | Better | Decision |
|--------|-------|------------|--------|----------|
| **MAE** | ₹1,754 | On average, every prediction is ₹1,754 away from the actual rent | ⬇️ Lower = Better | Train another model and compare |
| **RMSE > MAE** | Gap = ₹2,708 | Most predictions are ~₹1,754 off, but a few are severely wrong pulling RMSE up to ₹4,462 | ⬇️ Lower = Better | Train another model and compare |
| **R² Score** | 0.33 | Linear Regression model's prediction is weak ~67% of rent variation unexplained | ⬆️ Higher = Better | Train another model (tree-based) to improve |

---

> **Verdict:** Linear Regression is not good enough. Move to tree-based models (Random Forest / XGBoost) to capture non-linear rent patterns.

In [16]:
lr.coef_

array([[-0.07099695,  0.40381681,  0.44220164,  0.00606746, -0.00652418,
        -0.11000773,  0.13723976, -0.00303758,  0.25176793,  0.02357802,
        -0.09637525,  0.00417878,  0.15154267,  0.02307527, -0.04958345,
         0.03007933,  0.1287062 , -0.26377551,  0.01675561,  0.003946  ,
         0.10853681,  0.07348781, -0.11116017,  0.06455939,  0.06289336,
        -0.0180705 , -0.04482286,  0.04378457, -0.05731983,  0.06514942,
        -0.05161416, -0.03545369,  0.01008463,  0.02536906]])

---
# Train More Models to comapre against baseline
---

## **Ok now before trying tree based models, I'm going to try regularization with the linear model**

### L2 Regularization (Ridge Regression)

- Since i have many featues and corelating with each (some), linear regression may gave too much importance to some features
- The goal is to penalize those coefficients and potentially reduce the variance or overfitting 


In [17]:
X_train.head()

,latitude,longitude,locality,transit_score,lifestyle_score,occupancy,deposit,attached_bathroom,food_included,mess,...,gender_BOTH,gender_FEMALE,gender_MALE,parking_Bike,parking_Bike and Car,parking_Car,parking_No Parking,available_for_Anyone,available_for_Student,available_for_Working Professional
0,12.913368,80.228812,8.935934,6.3,6.3,2.0,8.294300,1,1,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,12.979819,80.242495,8.901274,8.0,7.9,2.0,8.006701,0,1,0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,12.989589,80.248599,8.886307,7.9,8.3,0.0,10.283669,0,0,1,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,12.926202,80.111700,8.876267,8.2,9.6,2.0,7.601402,0,1,0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,13.053022,80.213763,8.693086,6.7,6.3,2.0,10.308986,0,0,0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [6]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# before train a ridge model, i need to scale the features, cuz ridge is sensitive to features scale
# so basically ridge is used with standard scalar

scaler = StandardScaler()

# fit on train only ; fit means learn the rules on the data
scaler.fit(X_train)

# transform on train/test/val ; transform means, apply the learned rules on the data
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# to find the best alpha value
for alpha in [1, 5, 10, 40, 42, 43,44, 60]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    pred = ridge.predict(X_val_scaled)

    print(
        round(mean_absolute_error(y_val, (pred)), 4),
        round(root_mean_squared_error(y_val, pred), 4),
        round(r2_score(y_val, pred), 4)
    )

0.1993 0.303 0.3346
0.1991 0.3029 0.3352
0.1989 0.3027 0.3357
0.1988 0.3025 0.3367
0.1989 0.3025 0.3367
0.1989 0.3025 0.3367
0.1989 0.3025 0.3366
0.199 0.3026 0.3363


In [7]:
# model fit

alpha = 43 # found that the alpha = 43 is penalize value from the above iterations
ridge = Ridge(alpha=alpha)
ridge.fit(X_train_scaled, y_train)
result('Ridge rgression', ridge, X_val_scaled, y_val)
pd.DataFrame(all_results).T

,Log MAE,Log RMSE,Log R2,Actual MAE,Actual RMSE,Actual R2
Linear Regression,0.1994,0.3030,0.3345,1754.5806,4462.0860,0.1646
Ridge rgression,0.1989,0.3025,0.3367,1754.2632,4478.4207,0.1585


In [8]:
train_vs_val('Ridge', ridge, X_train_scaled, X_val_scaled, y_train, y_val)
pd.DataFrame(train_vs_val_result).T

,Train Score,Val Score,Fitting
Linear Regression,0.462222,0.334473,Weak
Ridge,0.459903,0.336659,Weak


## Ridge Regression result:

- alpha = ~43.0 gives the best result = 0.3367
- still similar to the linear regression, no drastical improvements


**Takeaway**: Ridge isn't dramatically better than Linear Regression for this dataset. That's actually a useful modeling finding.

---
# Lasso Regression

- since ridge isnt make any improvement
- here i see lasso, hope it could lead to anything

In [9]:
from sklearn.linear_model import Lasso

for alpha in [0.0001, 0.001, 0.005, 0.006, 0.1, 0.5, 1]:
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train_scaled, y_train)
    pred = lasso.predict(X_val_scaled)

    print(
        round(mean_absolute_error(y_val, pred), 4),
        round(root_mean_squared_error(y_val, pred), 4),
        round(r2_score(y_val, pred), 4)
    )

0.1992 0.3029 0.3349
0.1986 0.3024 0.3374
0.1986 0.3022 0.3382
0.1986 0.3024 0.3371
0.2473 0.3603 0.0593
0.2581 0.3717 -0.0015
0.2581 0.3717 -0.0015


In [10]:
# here alpha = 0.005 is the best hyperparametr, observed from the above iteration
lasso = Lasso(alpha=0.005)
lasso.fit(X_train_scaled, y_train)
result('lasso', lasso, X_val_scaled, y_val)
# lasso.fit(X_train_scaled, y_train)
# pred = lasso.predict(X_val_scaled)
pd.DataFrame(all_results).T

,Log MAE,Log RMSE,Log R2,Actual MAE,Actual RMSE,Actual R2
Linear Regression,0.1994,0.3030,0.3345,1754.5806,4462.0860,0.1646
Ridge rgression,0.1989,0.3025,0.3367,1754.2632,4478.4207,0.1585
lasso,0.1986,0.3022,0.3382,1754.8490,4482.2114,0.1570


In [11]:
train_vs_val('Lasso', lasso, X_train_scaled, X_val_scaled, y_train, y_val)
pd.DataFrame(train_vs_val_result).T

,Train Score,Val Score,Fitting
Linear Regression,0.462222,0.334473,Weak
Ridge,0.459903,0.336659,Weak
Lasso,0.451662,0.338156,Weak


## lasso result:

- Lasso gives me almost the same performance while removing some less useful features.

    ```md 
    Linear Regression → R² ≈ 0.3345
    Ridge             → R² ≈ 0.3367
    Lasso (α=0.005)   → R² ≈ 0.3382
    ```

- so far, **Conclusion**: Lasso is currently your best of the three, but the improvement is small. Now you've learned why Lasso is useful rather than just chasing a higher score.

## AND SKIPPING ELASTIC NET

> Elastic Net is just Ridge + lasso Combined, It finds a middle ground between the two.

- The problem was never regularization. The problem is that rent pricing is non-linear and linear models simply can't see that no matter how regularize them.

# Tree based Models

## 1. Decision Tree (single tree)

In [12]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

dt = DecisionTreeRegressor(max_depth=5, splitter='best')
dt.fit(X_train, y_train)

result('Decision Tree', dt, X_val, y_val)
pd.DataFrame(all_results).T

,Log MAE,Log RMSE,Log R2,Actual MAE,Actual RMSE,Actual R2
Linear Regression,0.1994,0.3030,0.3345,1754.5806,4462.0860,0.1646
Ridge rgression,0.1989,0.3025,0.3367,1754.2632,4478.4207,0.1585
lasso,0.1986,0.3022,0.3382,1754.8490,4482.2114,0.1570
Decision Tree,0.1914,0.3040,0.3301,1691.5174,4468.0908,0.1623


In [13]:
train_vs_val('Decision Tree', dt, X_train, X_val, y_train, y_val)
pd.DataFrame(train_vs_val_result).T

,Train Score,Val Score,Fitting
Linear Regression,0.462222,0.334473,Weak
Ridge,0.459903,0.336659,Weak
Lasso,0.451662,0.338156,Weak
Decision Tree,0.61467,0.330058,Over Fitting


# Result of single Tree (Decision TRee)

- Since it is an tree based model, we can see that training accuracy increases
- But it uses Single tree so it can not that effective (val score still ~same as linear models), so try random forest

# 2. Random Forest

In [14]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

result('Random Forest', rf, X_val, y_val)
train_vs_val('Random Forest', rf, X_train, X_val, y_train, y_val)
pd.DataFrame(all_results).T

d:\Hustle\Chennai-PG\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,Log MAE,Log RMSE,Log R2,Actual MAE,Actual RMSE,Actual R2
Linear Regression,0.1994,0.3030,0.3345,1754.5806,4462.0860,0.1646
Ridge rgression,0.1989,0.3025,0.3367,1754.2632,4478.4207,0.1585
lasso,0.1986,0.3022,0.3382,1754.8490,4482.2114,0.1570
Decision Tree,0.1914,0.3040,0.3301,1691.5174,4468.0908,0.1623
Random Forest,0.1600,0.2649,0.4914,1472.0468,4315.9519,0.2184


In [16]:
# Rough Comparision for train/val for random forest
rf_train_pred = rf.predict(X_train)
rf_val_pred = rf.predict(X_val)

print("Train R²:", round(r2_score(y_train, rf_train_pred), 4))
print("Val R²:", round(r2_score(y_val, rf_val_pred), 4))

Train R²: 0.9445
Val R²: 0.4914


## Results as of now

```md 
Train R² = 0.944   ← memorised training data
Val R²   = 0.491   ← falls apart on new data
Gap      = 0.453   ← massive overfit
```
- So we can see a drastic improvements in the scores,
- But the model over learn from the training, and perform poor on val
- So need to do Hyperparametr tuning

---
```md
max_depth        → how deep each tree can go (main culprit)
min_samples_leaf → minimum samples needed at a leaf (stops tiny splits)
min_samples_split→ minimum samples needed to split a node
max_features     → features considered per split (more = more overfit)
n_estimators     → number of trees (more = better, not overfit-related)
```
---

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [5, 10, 15, 20, None],
    'min_samples_leaf' : [1, 5, 10, 20],
    'max_features'     : ['sqrt', 'log2', 0.5]
}

rf_grid = GridSearchCV(
    estimator  = RandomForestRegressor(random_state=42),
    param_grid = param_grid,
    cv         = 5,           # 5-fold cross validation
    scoring    = 'r2',
    n_jobs     = -1,          # use all CPU cores
    verbose    = 2            # watch progress
)

rf_grid.fit(X_train, y_train)

result('rf_grid', rf_grid, X_val, y_val)

print("Best params:", rf_grid.best_params_)
print("Best CV R²:", rf_grid.best_score_)

Fitting 5 folds for each of 180 candidates, totalling 900 fits


d:\Hustle\Chennai-PG\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Best params: {'max_depth': 15, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}
Best CV R²: 0.6018749040331908


In [18]:
# Check Fitting resutls for random forest with GridSearchCV

train_vs_val('rf_grid', rf_grid, X_train, X_val, y_train, y_val)
pd.DataFrame(train_vs_val_result).T

,Train Score,Val Score,Fitting
Linear Regression,0.462222,0.334473,Weak
Ridge,0.459903,0.336659,Weak
Lasso,0.451662,0.338156,Weak
Decision Tree,0.61467,0.330058,Over Fitting
Random Forest,0.944501,0.491434,Over Fitting
rf_grid,0.939564,0.510541,Over Fitting


In [19]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 8, 10, 15, 20, None],
    'min_samples_leaf': [1, 5, 10, 20, 50],
    'min_samples_split': [2, 5, 20],
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 1.0]
}

rf_random = RandomizedSearchCV(
    estimator = RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=30, # 30 random combinations
    cv = 5,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

rf_random.fit(X_train, y_train)

result('rf_random', rf_random, X_val, y_val)

print(f'Best Params: {rf_random.best_params_}')
print(f'\nBest CV R2: {rf_random.best_score_}')

Fitting 5 folds for each of 30 candidates, totalling 150 fits


d:\Hustle\Chennai-PG\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Best Params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}

Best CV R2: 0.5806157545599013


In [20]:
# Check Fitting resutls for random forest with GridSearchCV

train_vs_val('rf_random', rf_random, X_train, X_val, y_train, y_val)
pd.DataFrame(train_vs_val_result).T

,Train Score,Val Score,Fitting
Linear Regression,0.462222,0.334473,Weak
Ridge,0.459903,0.336659,Weak
Lasso,0.451662,0.338156,Weak
Decision Tree,0.61467,0.330058,Over Fitting
Random Forest,0.944501,0.491434,Over Fitting
rf_grid,0.939564,0.510541,Over Fitting
rf_random,0.943005,0.482506,Over Fitting


In [21]:
# narrow, aggressive grid (another try of grid)
param_grid_constrained = {
    'n_estimators'     : [200, 300],
    'max_depth'        : [4, 5, 6, 7, 8],
    'min_samples_leaf' : [20, 30, 50, 75, 100],
    'max_features'     : ['sqrt', 0.3],
}

rf_constrained = GridSearchCV(
    estimator  = RandomForestRegressor(random_state=42),
    param_grid = param_grid_constrained,
    cv         = 5,
    scoring    = 'r2',
    n_jobs     = -1,
    verbose    = 1
)

rf_constrained.fit(X_train, y_train)

result('rf_grid_const', rf_constrained, X_val, y_val)

print("Best params :", rf_constrained.best_params_)
print("Best CV R²  :", rf_constrained.best_score_)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


d:\Hustle\Chennai-PG\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Best params : {'max_depth': 8, 'max_features': 0.3, 'min_samples_leaf': 20, 'n_estimators': 200}
Best CV R²  : 0.41902656172883657


In [22]:
# Final Score comparison till random forests
train_vs_val('rf_grid_const', rf_constrained, X_train, X_val, y_train, y_val)
pd.DataFrame(train_vs_val_result).T

,Train Score,Val Score,Fitting
Linear Regression,0.462222,0.334473,Weak
Ridge,0.459903,0.336659,Weak
Lasso,0.451662,0.338156,Weak
Decision Tree,0.61467,0.330058,Over Fitting
Random Forest,0.944501,0.491434,Over Fitting
rf_grid,0.939564,0.510541,Over Fitting
rf_random,0.943005,0.482506,Over Fitting
rf_grid_const,0.51671,0.395311,Weak


In [23]:
# Final Results
pd.DataFrame(all_results).T

,Log MAE,Log RMSE,Log R2,Actual MAE,Actual RMSE,Actual R2
Linear Regression,0.1994,0.3030,0.3345,1754.5806,4462.0860,0.1646
Ridge rgression,0.1989,0.3025,0.3367,1754.2632,4478.4207,0.1585
lasso,0.1986,0.3022,0.3382,1754.8490,4482.2114,0.1570
Decision Tree,0.1914,0.3040,0.3301,1691.5174,4468.0908,0.1623
Random Forest,0.1600,0.2649,0.4914,1472.0468,4315.9519,0.2184
rf_grid,0.1587,0.2599,0.5105,1452.6903,4278.7966,0.2318
rf_random,0.1689,0.2672,0.4825,1545.3338,4359.5994,0.2025
rf_grid_const,0.1916,0.2888,0.3953,1703.4043,4461.1543,0.1649


# Results Of Random Forest

---
```md
Random Forest      → R²=0.49, overfit
rf_grid            → R²=0.51, best RF, still overfit ← WINNER for now
rf_random          → R²=0.48, worse than grid
rf_grid_const      → R²=0.39, overcorrected, underfits
```
---

## Random Forest Conclusion
- Best RF: rf_grid (Val R²=0.510, gap=0.43)
- Persistent overfitting RF memorises noise in scraped data
- Constrained RF fixed gap but killed val score (0.395)
- Best validation R² observed so far is ~0.51 with Random Forest.
- Moving to XGBoost handles overfitting better via boosting + learning rate

## Progress and plan

```md                 
                 Models
                    │
       ┌────────────┴────────────┐
       ↓                         ↓
Linear models              Tree models
       │                         │
LR → Ridge → Lasso        DT → Random Forest
                                  │
                                  ↓
                         Hyperparameter tuning
                         Grid / RandomizedSearch
                                  │
                                  ↓
                              XGBoost
                                  │
                                  ↓
                         Compare all models
                                  │
                                  ↓
                         Select final model
                                  │
                                  ↓
                         Production Pipeline
```